# TensorFlow Classification: จากข้อมูลสู่การประเมินโมเดล

คงข้อมูลและโครงสร้างโมเดลเดิม: **2000 samples, 20 features** และ Neural Network แบบ **20 → 16 → 2**

เป้าหมายคือให้เห็น Flow ครบ: `Data → Split → Model → Training → Evaluation → Prediction`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('TensorFlow version:', tf.__version__)

## 1) สร้างข้อมูลแบบเดิม

แต่ละ sample มี input 20 ค่า และกำหนด target จากผลรวมของ input:

- ผลรวมมากกว่า 0 → Class 1
- ผลรวมน้อยกว่าหรือเท่ากับ 0 → Class 0

In [ ]:
x = np.random.randn(2000, 20).astype(np.float32)
y = (x.sum(axis=1) > 0).astype(np.int32)

print('Input shape :', x.shape)
print('Target shape:', y.shape)
print('Sample input:', np.round(x[0], 2))
print('Sum =', round(float(x[0].sum()), 3), '→ Target =', y[0])

counts = np.bincount(y)
plt.bar(['Class 0', 'Class 1'], counts, color=['#118ab2', '#ef476f'])
plt.title('Class distribution'); plt.ylabel('Number of samples')
for i, value in enumerate(counts): plt.text(i, value+10, str(value), ha='center')
plt.ylim(0, counts.max()*1.12); plt.show()

## 2) มองข้อมูล 20 มิติด้วย PCA
PCA ลดจาก 20 features เหลือ 2 แกนเพื่อใช้แสดงภาพเท่านั้น โมเดลยังคงเรียนจากข้อมูล 20 features จริง

In [ ]:
pca = PCA(n_components=2)
x_pca = pca.fit_transform(x)
plt.figure(figsize=(8, 6))
for class_id, color in [(0, '#118ab2'), (1, '#ef476f')]:
    mask = y == class_id
    plt.scatter(x_pca[mask,0], x_pca[mask,1], s=15, alpha=.55, c=color, label=f'Class {class_id}')
plt.title('20-dimensional data projected to 2D using PCA')
plt.xlabel('Principal Component 1'); plt.ylabel('Principal Component 2')
plt.legend(); plt.grid(alpha=.2); plt.show()
print('Variance visible in 2D:', f'{pca.explained_variance_ratio_.sum():.1%}')

## 3) แบ่ง Training / Validation / Testing

- Training: ใช้ปรับ weights
- Validation: ตรวจแนวโน้มระหว่าง training
- Testing: วัดผลครั้งสุดท้ายกับข้อมูลที่โมเดลไม่เคยใช้ฝึก

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.20, random_state=SEED, stratify=y
)
print('Training + Validation:', x_train.shape)
print('Testing              :', x_test.shape)
print('model.fit จะแบ่ง 20% ของ x_train ไปเป็น Validation อีกครั้ง')

## 4) สร้างโมเดลแบบเดิม

`20 inputs → 16 ReLU neurons → 2 Softmax outputs`

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(20,), name='Input_20_features'),
    tf.keras.layers.Dense(16, activation='relu', name='Hidden_ReLU'),
    tf.keras.layers.Dense(2, activation='softmax', name='Output_Softmax'),
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()
print('Total parameters = (20×16+16) + (16×2+2) = 370')

## 5) ดู Prediction ก่อน Training
ก่อนเรียน ค่า probability ยังมาจาก weights ที่สุ่มเริ่มต้น

In [ ]:
weights_before = [w.copy() for w in model.get_weights()]
prob_before = model.predict(x_test[:5], verbose=0)
for i, probability in enumerate(prob_before):
    print(f'Sample {i}: target={y_test[i]} | P(class 0)={probability[0]:.3f}, P(class 1)={probability[1]:.3f}')

## 6) Training
ในแต่ละ batch โมเดลทำ Forward propagation → Loss → Backpropagation → Adam update

In [ ]:
history = model.fit(
    x_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.20,
    verbose=1,
)

## 7) กราฟ Learning Curve
สังเกตว่า Loss ควรลดลง ส่วน Accuracy ควรเพิ่มขึ้น และ Training กับ Validation ไม่ควรแยกห่างกันมาก

In [ ]:
epochs = np.arange(1, len(history.history['loss']) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(epochs, history.history['loss'], 'o-', label='Training')
ax[0].plot(epochs, history.history['val_loss'], 'o-', label='Validation')
ax[0].set(title='Loss', xlabel='Epoch', ylabel='Cross-entropy'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(epochs, np.array(history.history['accuracy'])*100, 'o-', label='Training')
ax[1].plot(epochs, np.array(history.history['val_accuracy'])*100, 'o-', label='Validation')
ax[1].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy (%)', ylim=(40, 102)); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 8) ประเมินด้วย Testing Data

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
test_probability = model.predict(x_test, verbose=0)
test_prediction = np.argmax(test_probability, axis=1)
print(f'Test loss    : {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.2%}')
print('\nClassification report:\n', classification_report(y_test, test_prediction, digits=3))

In [ ]:
cm = confusion_matrix(y_test, test_prediction)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2): ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=15)
ax.set_xticks([0,1], ['Class 0','Class 1']); ax.set_yticks([0,1], ['Class 0','Class 1'])
ax.set(xlabel='Predicted', ylabel='Target', title='Confusion Matrix')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## 9) เปรียบเทียบ Target กับ Prediction บนภาพ PCA
PCA ใช้เพื่อวาดภาพเท่านั้น จึงอาจเห็นบางจุดซ้อนกันแม้โมเดลที่ใช้ข้อมูลครบ 20 มิติจะจำแนกได้ถูก

In [ ]:
test_pca = pca.transform(x_test)
fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for class_id, color in [(0,'#118ab2'), (1,'#ef476f')]:
    ax[0].scatter(*test_pca[y_test==class_id].T, s=18, alpha=.65, c=color, label=f'Class {class_id}')
    ax[1].scatter(*test_pca[test_prediction==class_id].T, s=18, alpha=.65, c=color, label=f'Class {class_id}')
ax[0].set_title('Actual Target'); ax[1].set_title('Model Prediction')
for a in ax: a.set_xlabel('PC1'); a.set_ylabel('PC2'); a.legend(); a.grid(alpha=.2)
plt.tight_layout(); plt.show()

## 10) ดู Probability รายตัวอย่าง
Softmax ให้ผลรวม probability เท่ากับ 1 และเลือกคลาสที่มี probability สูงสุด

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 6))
for i, ax in enumerate(axes.flat):
    ax.bar(['Class 0','Class 1'], test_probability[i], color=['#118ab2','#ef476f'])
    ax.set_ylim(0,1); ax.set_title(f'Target={y_test[i]} | Predict={test_prediction[i]}')
    ax.set_ylabel('Probability'); ax.grid(axis='y', alpha=.25)
plt.tight_layout(); plt.show()

## 11) Backpropagation เปลี่ยน Weight จริงหรือไม่?

In [ ]:
weights_after = model.get_weights()
change = weights_after[0] - weights_before[0]
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
v = max(abs(weights_before[0]).max(), abs(weights_after[0]).max())
ax[0].imshow(weights_before[0], cmap='coolwarm', aspect='auto', vmin=-v, vmax=v); ax[0].set_title('Weights before training')
ax[1].imshow(weights_after[0], cmap='coolwarm', aspect='auto', vmin=-v, vmax=v); ax[1].set_title('Weights after training')
ax[2].imshow(change, cmap='PiYG', aspect='auto'); ax[2].set_title('Weight change from learning')
for a in ax: a.set_xlabel('Hidden neuron'); a.set_ylabel('Input feature')
plt.tight_layout(); plt.show()
print('Mean absolute weight change:', float(np.mean(np.abs(change))))

## สรุปสำหรับผู้เรียน

1. Input 20 features ถูกส่งเข้า Neural Network
2. Forward propagation คำนวณ probability ของ Class 0 และ Class 1
3. Loss เปรียบเทียบ prediction กับ target
4. Backpropagation คำนวณ gradient ย้อนกลับ
5. Adam ใช้ gradient ปรับ weights
6. Validation ใช้ติดตามระหว่างเรียน ส่วน Testing ใช้ประเมินผลสุดท้าย